In [2]:
import os
import pandas as pd

# ==========================================
# 1. LOAD MODULAR FEATURE TABLES
# ==========================================
data_dir = '../modelData'

# Load each feature store, keeping loan_id as the index
loan_df = pd.read_csv(os.path.join(data_dir, 'features_loan.csv'), index_col='loan_id')
orders_df = pd.read_csv(os.path.join(data_dir, 'features_orders.csv'), index_col='loan_id')
trans_df = pd.read_csv(os.path.join(data_dir, 'features_transaction.csv'), index_col='loan_id')
client_df = pd.read_csv(os.path.join(data_dir, 'feature_client.csv'), index_col='loan_id') # Updated for client/co-signer features

# Ensure index is string type for clean joins
loan_df.index = loan_df.index.astype(str)
orders_df.index = orders_df.index.astype(str)
trans_df.index = trans_df.index.astype(str)
client_df.index = client_df.index.astype(str) # Ensure string index for client features

# ==========================================
# 2. JOIN ALL TABLES ON loan_id
# ==========================================
# Start with loan baseline (contains contract terms and target)
master_df = loan_df.join(orders_df, how='left')

# Join transaction features
master_df = master_df.join(trans_df, how='left')

# Join client/co-signer features (includes has_co_signer)
master_df = master_df.join(client_df, how='left')

# For missing features across tables, fill with 0
master_df = master_df.fillna(0)

# ==========================================
# 3. SPLIT INTO FEATURES (X) AND TARGET (y)
# ==========================================
if 'target' in master_df.columns:
    y = master_df['target']
    X = master_df.drop(columns=['target'])
else:
    raise ValueError("Target column not found in master dataframe!")

# Optional: Export the combined master dataset for direct model tracking
master_output_path = os.path.join(data_dir, 'master_modeling_matrix.csv')
master_df.to_csv(master_output_path)

# ==========================================
# 4. VALIDATION READOUT
# ==========================================
print("=" * 50)
print(" MASTER MODELING DATASET CREATED SUCCESSFULLY")
print(f" Total Loans (Rows):     {master_df.shape[0]}")
print(f" Total Features (Cols):  {X.shape[1]}")
print(f" Missing Values (NaN):   {master_df.isna().sum().sum()}")
print(f" Default Class Balance:  {y.sum()} defaults out of {len(y)} loans ({y.mean()*100:.2f}%)")
print(f" Saved to:               {master_output_path}")
print("=" * 50)

 MASTER MODELING DATASET CREATED SUCCESSFULLY
 Total Loans (Rows):     682
 Total Features (Cols):  31
 Missing Values (NaN):   0
 Default Class Balance:  76 defaults out of 682 loans (11.14%)
 Saved to:               ../modelData\master_modeling_matrix.csv


In [3]:
master_df.columns

Index(['amount', 'duration', 'payments', 'target', 'order_count',
       'order_total_amount', 'order_mean_amount', 'order_max_amount',
       'order_amt_household_utility', 'order_amt_insurance_payment',
       'order_amt_loan_payment', 'order_amt_other_unknown', 'is_closed',
       'loan_date', 'min_balance', 'mean_balance', 'std_balance',
       'total_tx_count', 'history_days', 'total_inflow', 'total_outflow',
       'mean_balance_90d', 'freq_negative_balance', 'last_balance',
       'sanction_interest', 'cash_reliance_ratio', 'months_history',
       'avg_monthly_inflow', 'tx_per_month', 'dsti_ratio', 'balance_trend_90d',
       'has_co_signer'],
      dtype='object')